<a href="https://colab.research.google.com/github/Nawaf-Rayhan585/YOLO_Projects/blob/main/ppe-safety-compliance/train_ppe_detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PPE Safety Compliance Detector — Train on Colab T4, Use Anywhere

Trains a YOLOv8 model to detect hard hats, safety vests, masks etc., **and**
their absence (classes like `NO-Hardhat`, `NO-Safety Vest`). `ppe_compliance.py`
treats any class whose name contains "no-", "no_", "without" or "missing"
as a violation, so most public PPE datasets work without any code changes.

**How to use:**
1. Runtime -> Change runtime type -> GPU -> T4
2. Run cells top to bottom
3. Download `best.pt` and drop it into this folder

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Install packages

In [ ]:
!pip install ultralytics roboflow -q

## 3. Get a PPE / construction-site-safety dataset

Search "PPE detection" or "construction site safety" on
[Roboflow Universe](https://universe.roboflow.com) — there are large public
datasets with classes like `Hardhat`, `NO-Hardhat`, `Safety Vest`,
`NO-Safety Vest`, `Mask`, `NO-Mask`. Download in **YOLOv8** format.

In [ ]:
from roboflow import Roboflow

ROBOFLOW_API_KEY = "YOUR_API_KEY"
WORKSPACE = "YOUR_WORKSPACE"
PROJECT = "YOUR_PROJECT"
VERSION = 1

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
dataset = project.version(VERSION).download("yolov8")

print(dataset.location)

## 4. Train

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=80,
    imgsz=640,
    batch=16,
    device=0,
    project="ppe_detector",
    name="train1",
    plots=True,
)

## 5. Validate

In [ ]:
metrics = model.val()
print(metrics)

## 6. Download your trained model

In [ ]:
from google.colab import files

files.download("ppe_detector/train1/weights/best.pt")

## 7. Use it locally

```bash
pip install -r requirements.txt
python ppe_compliance.py --model best.pt --source site.mp4
```